# Chapter 12: Exception Handling and Recovery

EliteAI Academy

Handle failures with retries, fallbacks, validation, degradation, rollback, self-correction, and human escalation.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re, time, random, logging
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple
from enum import Enum
from datetime import datetime

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL
    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.", max_tokens: int = 512) -> str:
        r = _client.messages.create(thinking={"type": "disabled"}, model=_model, max_tokens=max_tokens, system=system,
                                    messages=[{"role": "user", "content": prompt}])
        return r.content[0].text.strip()

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")
    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.", max_tokens: int = 512) -> str:
        return _gmodel.generate_content(f"{system}\n\n{prompt}").text.strip()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("AgentExceptions")

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply: EXCEPTION HANDLER READY"))

## 1. Custom Exception Hierarchy

In [ ]:
class AgentError(Exception):
    'Base class for all agent errors.'
    def __init__(self, message: str, recoverable: bool = True, details: dict = None):
        super().__init__(message)
        self.recoverable = recoverable
        self.details     = details or {}
        self.timestamp   = datetime.now().isoformat()

class ToolError(AgentError):        'Tool call failed or returned invalid output.'
class APIError(AgentError):         'External API call failed.'
class TimeoutError(AgentError):     'Operation exceeded time limit.'
class ValidationError(AgentError):  'Output failed schema/format validation.'
class ParseError(AgentError):       'Could not parse LLM output as expected format.'
class EscalationRequired(AgentError):
    'Problem too severe — requires human intervention.'
    def __init__(self, message, ticket_id: str = None):
        super().__init__(message, recoverable=False)
        self.ticket_id = ticket_id or f"ESC-{random.randint(1000,9999)}"

@dataclass
class ErrorRecord:
    'Structured error log entry.'
    error_type:  str
    message:     str
    context:     str
    recoverable: bool
    recovery:    str = "none"
    timestamp:   str = field(default_factory=lambda: datetime.now().isoformat()[:19])

    def __str__(self):
        icon = "🔄" if self.recoverable else "🚨"
        return f"{icon} [{self.timestamp}] {self.error_type}: {self.message} | recovery={self.recovery}"

class ErrorLog:
    'Central error registry with summary reporting.'
    def __init__(self):
        self.records: List[ErrorRecord] = []

    def log(self, error: AgentError, context: str, recovery: str = "none"):
        r = ErrorRecord(
            error_type  = type(error).__name__,
            message     = str(error),
            context     = context,
            recoverable = error.recoverable,
            recovery    = recovery
        )
        self.records.append(r)
        level = logging.WARNING if error.recoverable else logging.ERROR
        logger.log(level, str(r))

    def summary(self):
        total     = len(self.records)
        recovered = sum(1 for r in self.records if r.recovery != "none")
        critical  = sum(1 for r in self.records if not r.recoverable)
        print(f"\n📋 Error Log: {total} errors | {recovered} recovered | {critical} critical")
        for r in self.records:
            print(f"  {r}")

error_log = ErrorLog()
print("✅ Exception hierarchy and ErrorLog ready")

## 2. Strategy 1 — Retry with Exponential Backoff

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

class UnreliableWeatherAPI:
    'Simulates a flaky external API that fails 60% of the time.'
    call_count = 0

    @classmethod
    def get_weather(cls, city: str) -> dict:
        cls.call_count += 1
        print(f"  🌐 API call #{cls.call_count} for '{city}'...")
        if random.random() < 0.6:
            raise APIError(f"503 Service Unavailable for city='{city}'", recoverable=True)
        return {"city": city, "temp": f"{random.randint(15,35)}°C", "condition": "Sunny"}

def retry_with_backoff(fn: Callable, max_attempts: int = 4,
                       base_delay: float = 0.2, context: str = "") -> Any:
    'Retry a function with exponential backoff.'
    last_error = None
    for attempt in range(max_attempts):
        try:
            return fn()
        except AgentError as e:
            last_error = e
            if not e.recoverable:
                raise
            delay = base_delay * (2 ** attempt) + random.uniform(0, 0.1)
            print(f"  ⚠️  Attempt {attempt+1} failed: {e}. Retrying in {delay:.2f}s...")
            error_log.log(e, context, recovery=f"retry_{attempt+2}")
            time.sleep(delay)
    raise last_error

@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=0.2, min=0.1, max=2),
    retry=retry_if_exception_type(APIError),
    reraise=True
)
def get_weather_with_retry(city: str) -> dict:
    return UnreliableWeatherAPI.get_weather(city)

print("=== RETRY WITH EXPONENTIAL BACKOFF ===")

print("\n[Manual retry]")
UnreliableWeatherAPI.call_count = 0
try:
    result = retry_with_backoff(
        fn=lambda: UnreliableWeatherAPI.get_weather("Lagos"),
        max_attempts=4, base_delay=0.1, context="weather_lookup"
    )
    print(f"  ✅ Success: {result}")
except AgentError as e:
    print(f"  ❌ All retries exhausted: {e}")

print("\n[Tenacity decorator]")
UnreliableWeatherAPI.call_count = 0
try:
    result = get_weather_with_retry("London")
    print(f"  ✅ Success: {result}")
except APIError as e:
    print(f"  ❌ All retries exhausted: {e}")

## 3. Strategy 2 — Fallback Chain

Primary tool fails → try fallback → try degraded → last resort default.

In [ ]:
class FallbackChain:
    'Try tools in priority order. Returns the first success.'
    def __init__(self, handlers: List[Tuple[str, Callable]]):
        self.handlers = handlers

    def execute(self, *args, **kwargs) -> Tuple[Any, str]:
        for name, handler in self.handlers:
            try:
                print(f"  → Trying [{name}]...")
                result = handler(*args, **kwargs)
                print(f"    ✅ [{name}] succeeded")
                return result, name
            except Exception as e:
                print(f"    ❌ [{name}] failed: {type(e).__name__}: {e}")
                if isinstance(e, AgentError):
                    error_log.log(e, name, recovery="fallback")

        raise AgentError("All fallbacks exhausted", recoverable=False)

def precise_location_lookup(address: str) -> dict:
    'High precision geocoding — often fails.'
    if random.random() < 0.7:
        raise ToolError(f"Geocoding API unavailable for '{address}'", recoverable=True)
    return {"address": address, "lat": 6.5244, "lng": 3.3792, "precision": "exact"}

def general_area_lookup(address: str) -> dict:
    'City-level lookup — more reliable.'
    if random.random() < 0.3:
        raise ToolError(f"Area lookup failed for '{address}'", recoverable=True)
    city = address.split(",")[0].strip()
    return {"city": city, "region": "West Africa", "precision": "city"}

def llm_location_estimate(address: str) -> dict:
    'LLM-based fallback — always available.'
    region = call_llm(
        f"What country/region is this address in? '{address}' — reply in 5 words or less."
    )
    return {"address": address, "region": region, "precision": "estimated"}

def static_default(address: str) -> dict:
    'Last-resort default — always succeeds.'
    return {"address": address, "region": "Unknown", "precision": "none"}

location_chain = FallbackChain([
    ("PreciseGeocoding",  precise_location_lookup),
    ("GeneralAreaLookup", general_area_lookup),
    ("LLMEstimate",       llm_location_estimate),
    ("StaticDefault",     static_default),
])

print("=== FALLBACK CHAIN ===")
for address in ["123 Marina, Lagos, Nigeria", "456 Tech Hub, Abuja"]:
    print(f"\n📍 Looking up: '{address}'")
    try:
        result, source = location_chain.execute(address)
        print(f"  Result (via {source}): {result}")
    except AgentError as e:
        print(f"  ❌ Critical failure: {e}")

## 4. Strategy 3 — Output Validation and Parse Error Recovery

In [ ]:
def safe_parse_json(raw: str, schema_keys: List[str], max_retries: int = 2) -> dict:
    'Parse JSON from LLM output with:'
    def _parse_attempt(text: str) -> dict:
        clean = re.sub(r"```[\w]*", "", text).strip()

        clean = re.sub(r",\s*([}\]])", r"\1", clean)
        data  = json.loads(clean)

        missing = [k for k in schema_keys if k not in data]
        if missing:
            raise ValidationError(f"Missing required keys: {missing}", recoverable=True)
        return data

    try:
        return _parse_attempt(raw)
    except (json.JSONDecodeError, ValidationError) as e:
        error_log.log(
            ParseError(str(e)) if isinstance(e, json.JSONDecodeError) else e,
            "json_parse", recovery="llm_repair"
        )

    for attempt in range(max_retries):
        print(f"  🔧 LLM repair attempt {attempt+1}...")
        repaired = call_llm(
            f"""Fix this malformed JSON. Required keys: {schema_keys}
Return ONLY valid JSON with all required keys.

Malformed input:
{raw}""",
            system="You are a JSON repair specialist. Return only valid JSON."
        )
        try:
            return _parse_attempt(repaired)
        except Exception as e:
            raw = repaired

    raise ParseError(f"Could not parse JSON after {max_retries} repair attempts", recoverable=False)

test_cases = [
    ('{"name": "Phunkie", "role": "Data Engineer", "company": "Wragby"}',
     ["name", "role", "company"]),
    ('```json\n{"name": "Phunkie", "role": "Data Engineer",}\n```',
     ["name", "role", "company"]),
    ('The user is named Phunkie and works as a data engineer at Wragby.',
     ["name", "role", "company"]),
]

print("=== JSON VALIDATION & PARSE RECOVERY ===")
for raw, schema in test_cases:
    print(f"\n📥 Input: {raw[:70]}...")
    try:
        result = safe_parse_json(raw, schema)
        print(f"  ✅ Parsed: {result}")
    except ParseError as e:
        print(f"  ❌ {e}")

## 5. Strategy 4 — Graceful Degradation

When a capability is unavailable, degrade to a simpler version instead of failing completely.

In [ ]:
class DegradationLevel(Enum):
    FULL       = "full"
    REDUCED    = "reduced"
    MINIMAL    = "minimal"
    EMERGENCY  = "emergency"

class GracefulDegradationAgent:
    'Agent that automatically degrades capabilities when tools fail.'
    def __init__(self):
        self.level         = DegradationLevel.FULL
        self.failed_tools: List[str] = []
        self.error_log     = ErrorLog()

    def _try_tool(self, tool_name: str, fn: Callable, *args) -> Optional[Any]:
        if tool_name in self.failed_tools:
            return None
        try:
            return fn(*args)
        except Exception as e:
            self.failed_tools.append(tool_name)
            self.error_log.log(
                ToolError(str(e)),
                context=tool_name,
                recovery="graceful_degradation"
            )
            self._update_degradation_level()
            return None

    def _update_degradation_level(self):
        n = len(self.failed_tools)
        if   n >= 4: self.level = DegradationLevel.EMERGENCY
        elif n >= 3: self.level = DegradationLevel.MINIMAL
        elif n >= 1: self.level = DegradationLevel.REDUCED
        else:        self.level = DegradationLevel.FULL

    def handle_travel_query(self, query: str) -> dict:
        'Travel assistant that gracefully degrades when tools fail.'

        def flight_search(q):
            if random.random() < 0.7: raise ToolError("Flight API down")
            return {"flights": ["BA001 - £450", "EK201 - £620"]}

        def hotel_search(q):
            if random.random() < 0.5: raise ToolError("Hotel API timeout")
            return {"hotels": ["Marriott £180/night", "Hilton £220/night"]}

        def weather_check(q):
            if random.random() < 0.4: raise ToolError("Weather API rate limited")
            return {"weather": "Sunny, 28°C"}

        flights = self._try_tool("flight_search", flight_search, query)
        hotels  = self._try_tool("hotel_search",  hotel_search,  query)
        weather = self._try_tool("weather_check", weather_check, query)

        available_info = {k: v for k, v in
                         {"flights": flights, "hotels": hotels, "weather": weather}.items()
                         if v is not None}

        if not available_info:

            response = call_llm(
                f"Provide general travel advice for: {query} (no real-time data available)"
            )
        else:
            response = call_llm(
                f"Answer this travel query using available data. Note any missing info.\n"
                f"Query: {query}\nAvailable: {json.dumps(available_info)}"
            )

        if self.failed_tools:
            warning = f"⚠️  Note: {', '.join(self.failed_tools)} unavailable — using general advice for those."
        else:
            warning = None

        return {
            "level":          self.level.value,
            "response":       response,
            "warning":        warning,
            "failed_tools":   self.failed_tools,
            "available_data": list(available_info.keys())
        }

print("=== GRACEFUL DEGRADATION ===")
travel_agent = GracefulDegradationAgent()
result = travel_agent.handle_travel_query("Plan a trip from Lagos to London next month")

print(f"\nDegradation level: {result['level'].upper()}")
print(f"Available data:    {result['available_data']}")
print(f"Failed tools:      {result['failed_tools']}")
if result["warning"]:
    print(f"\n{result['warning']}")
print(f"\nResponse:\n{result['response']}")

## 6. Strategy 5 — State Rollback

In [ ]:
import copy

class TransactionalStateManager:
    'Agent state with checkpoint/rollback support.'
    def __init__(self, initial_state: dict):
        self._state      = copy.deepcopy(initial_state)
        self._checkpoints: List[Tuple[str, dict]] = [("initial", copy.deepcopy(initial_state))]

    @property
    def state(self) -> dict:
        return self._state

    def checkpoint(self, label: str):
        self._checkpoints.append((label, copy.deepcopy(self._state)))
        print(f"  💾 Checkpoint: '{label}' (total: {len(self._checkpoints)})")

    def rollback(self, steps: int = 1) -> str:
        if len(self._checkpoints) <= steps:
            steps = len(self._checkpoints) - 1
        label, state = self._checkpoints[-(steps+1)]
        self._state = copy.deepcopy(state)

        self._checkpoints = self._checkpoints[:-(steps)]
        print(f"  ↩️  Rolled back to checkpoint: '{label}'")
        return label

    def update(self, updates: dict):
        self._state.update(updates)

    def history(self):
        print("\n📜 Checkpoint history:")
        for label, state in self._checkpoints:
            print(f"  [{label}]: balance=${state.get('balance',0):.2f} | status={state.get('status','?')}")

def execute_financial_transaction(state_mgr: TransactionalStateManager,
                                   steps: List[Tuple[str, dict, bool]]):
    'Execute a multi-step financial operation.'
    for step_name, update, will_fail in steps:
        state_mgr.checkpoint(f"before_{step_name}")
        print(f"\n  [Step: {step_name}]")

        try:
            if will_fail:
                raise ToolError(f"{step_name} failed — external service error", recoverable=True)
            state_mgr.update(update)
            print(f"    ✅ State: {state_mgr.state}")

        except ToolError as e:
            print(f"    ❌ Error: {e}")
            rolled = state_mgr.rollback(steps=1)
            print(f"    ↩️  State restored to: {state_mgr.state}")
            error_log.log(e, context=step_name, recovery=f"rollback_to_{rolled}")
            return False

    return True

print("=== STATE ROLLBACK ===")

initial = {"balance": 1000.00, "status": "idle", "transaction_id": None}
sm = TransactionalStateManager(initial)
print(f"\nInitial state: {sm.state}")

steps = [
    ("debit_account",     {"balance": 750.00, "status": "debited"},                  False),
    ("transfer_funds",    {"balance": 750.00, "status": "transferred",
                           "transaction_id": "TXN-001"},                              True),
    ("notify_recipient",  {"status": "complete", "transaction_id": "TXN-001"},       False),
]

success = execute_financial_transaction(sm, steps)
print(f"\n{'✅ Transaction complete' if success else '❌ Transaction failed — state rolled back'}")
print(f"Final state: {sm.state}")
sm.history()

## 7. Strategy 6 — Self-Correction via Reflection

In [ ]:
class SelfCorrectingAgent:
    'Agent that diagnoses its own failures and generates'
    def __init__(self, role: str, max_self_corrections: int = 2):
        self.role              = role
        self.max_corrections   = max_self_corrections
        self.correction_log:   List[dict] = []

    def execute(self, task: str, adjusted_prompt: str = "") -> str:
        prompt = adjusted_prompt or task
        return call_llm(prompt, system=f"You are a {self.role}.")

    def diagnose(self, task: str, output: str, error: str) -> str:
        'LLM diagnoses the failure and suggests a corrected approach.'
        return call_llm(
            f"""You attempted a task but the output was invalid.

Task: {task}
Output: {output[:300]}
Error: {error}

Diagnose the problem and rewrite the task prompt to produce a correct output.
Return ONLY the corrected task prompt.""",
            system="You are a self-correction specialist. Fix the prompt to resolve the error."
        )

    def validate(self, output: str, expected_keys: List[str]) -> Tuple[bool, str]:
        'Validate that output is JSON with required keys.'
        try:
            data    = json.loads(re.sub(r"```[\w]*", "", output).strip())
            missing = [k for k in expected_keys if k not in data]
            if missing:
                return False, f"Missing keys: {missing}"
            return True, ""
        except json.JSONDecodeError as e:
            return False, f"JSON parse error: {e}"

    def run(self, task: str, expected_keys: List[str]) -> dict:
        'Execute with self-correction loop.'
        prompt = task
        output = ""

        for attempt in range(self.max_corrections + 1):
            print(f"\n[Attempt {attempt+1}]")
            output = self.execute(task, prompt)
            valid, error = self.validate(output, expected_keys)

            if valid:
                print(f"  ✅ Valid output produced")
                return json.loads(re.sub(r"```[\w]*", "", output).strip())

            print(f"  ❌ Validation failed: {error}")

            if attempt < self.max_corrections:
                print(f"  🔍 Diagnosing and correcting...")
                corrected_prompt = self.diagnose(task, output, error)
                self.correction_log.append({
                    "attempt": attempt + 1, "error": error,
                    "original_prompt": prompt[:100],
                    "corrected_prompt": corrected_prompt[:100]
                })
                prompt = corrected_prompt

        error_log.log(
            ValidationError(f"Self-correction failed after {self.max_corrections} attempts"),
            context=task[:60], recovery="escalation"
        )
        raise ValidationError(f"Could not produce valid output after {self.max_corrections} self-corrections")

print("=== SELF-CORRECTING AGENT ===")

agent = SelfCorrectingAgent(role="data extraction specialist", max_self_corrections=2)

task = """Extract these fields from the text below as a JSON object:
Required keys: customer_name, order_id, product, amount_usd, delivery_date

Text: 'John Smith placed order #ORD-5521 for a MacBook Pro at $2,499. 
Expected delivery is 15th June 2025.'"""

try:
    result = agent.run(task, ["customer_name", "order_id", "product", "amount_usd", "delivery_date"])
    print(f"\n✅ Final result: {json.dumps(result, indent=2)}")
    if agent.correction_log:
        print(f"\n📝 Self-corrections applied: {len(agent.correction_log)}")
except ValidationError as e:
    print(f"\n❌ {e}")

## 8. Strategy 7 — Escalation to Human

In [ ]:
@dataclass
class EscalationTicket:
    ticket_id:    str
    priority:     str
    summary:      str
    context:      dict
    assigned_to:  str
    created_at:   str = field(default_factory=lambda: datetime.now().isoformat()[:19])

    def display(self):
        print(f"\n🎫 ESCALATION TICKET {self.ticket_id}")
        print(f"   Priority:    {self.priority}")
        print(f"   Assigned to: {self.assigned_to}")
        print(f"   Summary:     {self.summary}")
        print(f"   Context:     {self.context}")
        print(f"   Created:     {self.created_at}")

class EscalationManager:
    'Manages escalation routing based on error type and severity.'

    ROUTING_RULES = {
        "APIError":          ("P2", "infrastructure-team@company.com"),
        "ValidationError":   ("P3", "data-quality-team@company.com"),
        "ToolError":         ("P2", "engineering-team@company.com"),
        "TimeoutError":      ("P2", "sre-team@company.com"),
        "EscalationRequired":("P1", "on-call-engineer@company.com"),
    }

    def __init__(self):
        self.tickets: List[EscalationTicket] = []

    def escalate(self, error: AgentError, context: dict) -> EscalationTicket:
        error_type = type(error).__name__
        priority, assignee = self.ROUTING_RULES.get(error_type, ("P3", "support@company.com"))

        summary = call_llm(
            f"Summarise this agent failure in one sentence for a support ticket:\nError: {error}\nContext: {context}",
            system="You are a concise technical writer. Write for an on-call engineer."
        )

        ticket = EscalationTicket(
            ticket_id   = f"ESC-{random.randint(10000,99999)}",
            priority    = priority,
            summary     = summary,
            context     = context,
            assigned_to = assignee
        )
        self.tickets.append(ticket)
        ticket.display()
        return ticket

    def summary(self):
        print(f"\n📊 Escalation summary: {len(self.tickets)} tickets")
        for p in ["P1", "P2", "P3"]:
            count = sum(1 for t in self.tickets if t.priority == p)
            print(f"  {p}: {count} ticket(s)")

print("=== ESCALATION MANAGER ===")
esc_mgr = EscalationManager()

errors = [
    (APIError("Payment gateway returned 502 for 10 consecutive calls"), {"service": "payments", "retries": 10}),
    (EscalationRequired("Customer data corruption detected in transaction batch"), {"batch_id": "B-4421", "records_affected": 847}),
    (TimeoutError("Database query exceeded 30s timeout", recoverable=False), {"query": "SELECT * FROM orders", "timeout_s": 30}),
]

for error, context in errors:
    esc_mgr.escalate(error, context)

esc_mgr.summary()

## 9. Full Resilient Agent — All Strategies Combined

In [ ]:
class ResilientAgent:
    'Production-grade resilient agent combining:'
    def __init__(self, name: str):
        self.name        = name
        self.error_log   = ErrorLog()
        self.escalation  = EscalationManager()
        self.state_mgr   = TransactionalStateManager({"status": "idle", "data": {}})

    def process_order(self, order: dict) -> dict:
        'Process an e-commerce order with full resilience.'
        print(f"\n{'='*55}")
        print(f"RESILIENT AGENT: Processing order {order.get('id', 'unknown')}")
        print(f"{'='*55}")

        self.state_mgr.checkpoint("order_received")
        self.state_mgr.update({"status": "processing", "data": {"order": order}})

        print("\n[Step 1] Checking inventory...")
        try:
            inventory = retry_with_backoff(
                fn=lambda: self._check_inventory(order["product_id"]),
                max_attempts=3, base_delay=0.05, context="inventory_check"
            )
            self.state_mgr.checkpoint("inventory_confirmed")
            print(f"  ✅ Inventory: {inventory}")
        except AgentError as e:
            self.error_log.log(e, "inventory", recovery="fallback")
            inventory = {"available": True, "source": "cache"}
            print(f"  ⚠️  Using cached inventory data")

        print("\n[Step 2] Processing payment...")
        self.state_mgr.checkpoint("before_payment")
        try:
            payment = self._process_payment(order["amount"])
            self.state_mgr.update({"data": {**self.state_mgr.state["data"], "payment": payment}})
            print(f"  ✅ Payment: {payment}")
        except ToolError as e:
            self.error_log.log(e, "payment", recovery="rollback")
            self.state_mgr.rollback(steps=1)
            print(f"  ↩️  Payment failed — rolled back. Escalating...")
            self.escalation.escalate(e, {"order": order, "state": self.state_mgr.state})
            return {"status": "failed", "reason": str(e), "escalated": True}

        print("\n[Step 3] Generating confirmation...")
        try:
            agent = SelfCorrectingAgent("order confirmation writer", max_self_corrections=2)
            confirmation = agent.run(
                f"Generate an order confirmation JSON for: {json.dumps(order)}. Required keys: order_id, status, message, estimated_delivery",
                ["order_id", "status", "message", "estimated_delivery"]
            )
            print(f"  ✅ Confirmation: {confirmation}")
        except ValidationError as e:
            self.error_log.log(e, "confirmation", recovery="graceful_degradation")
            confirmation = {"order_id": order.get("id"), "status": "processing",
                           "message": "Order received. Confirmation email coming shortly.",
                           "estimated_delivery": "3-5 business days"}
            print(f"  ⚠️  Using default confirmation template")

        self.state_mgr.update({"status": "complete"})
        self.state_mgr.checkpoint("order_complete")

        return {"status": "success", "confirmation": confirmation, "state": self.state_mgr.state}

    def _check_inventory(self, product_id: str) -> dict:
        if random.random() < 0.5: raise ToolError("Inventory API timeout")
        return {"product_id": product_id, "available": True, "qty": random.randint(1, 50)}

    def _process_payment(self, amount: float) -> dict:
        if random.random() < 0.25: raise ToolError("Payment gateway error: declined")
        return {"transaction_id": f"TXN-{random.randint(10000,99999)}", "amount": amount, "status": "approved"}

resilient = ResilientAgent("OrderProcessor")
result = resilient.process_order({
    "id": "ORD-7821", "product_id": "SKU-MAC-PRO",
    "amount": 2499.00, "customer": "Phunkie"
})
print(f"\n{'='*55}")
print(f"Final result: {result.get('status')}")
resilient.error_log.summary()

## 10. Quick Reference

In [ ]:
reference = """
╔══════════════════════╦════════════════════════╦══════════════════════════════╗
║ Strategy             ║ When to Use            ║ Key Code Pattern             ║
╠══════════════════════╬════════════════════════╬══════════════════════════════╣
║ Retry + Backoff      ║ Transient failures     ║ tenacity @retry or           ║
║                      ║ (network, rate limits) ║ exponential sleep loop       ║
╠══════════════════════╬════════════════════════╬══════════════════════════════╣
║ Fallback Chain       ║ Tool unavailable       ║ FallbackChain([(name, fn)])  ║
║                      ║ Need degraded path     ║ try each in order            ║
╠══════════════════════╬════════════════════════╬══════════════════════════════╣
║ Output Validation    ║ JSON format errors     ║ safe_parse_json + LLM repair ║
║ + LLM Repair         ║ Schema violations      ║                              ║
╠══════════════════════╬════════════════════════╬══════════════════════════════╣
║ Graceful Degradation ║ Multiple tool failures ║ DegradationLevel enum +      ║
║                      ║ Partial data OK        ║ skip known-failed tools      ║
╠══════════════════════╬════════════════════════╬══════════════════════════════╣
║ State Rollback       ║ Multi-step transaction ║ checkpoint() before each     ║
║                      ║ Partial writes danger  ║ step, rollback() on failure  ║
╠══════════════════════╬════════════════════════╬══════════════════════════════╣
║ Self-Correction      ║ Prompt/format issues   ║ diagnose() → rewrite prompt  ║
║                      ║ Repeatable LLM errors  ║ → retry with corrected prompt║
╠══════════════════════╬════════════════════════╬══════════════════════════════╣
║ Escalation           ║ Unrecoverable errors   ║ EscalationManager.escalate() ║
║                      ║ Requires human         ║ priority routing + LLM       ║
║                      ║                        ║ summary for ticket           ║
╚══════════════════════╩════════════════════════╩══════════════════════════════╝

⚠️  ALWAYS: Log every error with (type, message, context, recovery_action)
⚠️  ALWAYS: Set max_retries to prevent infinite loops
⚠️  ALWAYS: Mark errors as recoverable=True/False to control flow
"""
print(reference)

## 11. Exercises

- **Exercise 1 — Circuit Breaker:** Implement a `CircuitBreaker` class. After 3 consecutive failures, it opens the circuit (stops calls for 30s). After the cooldown, it allows one test call (half-open). If successful, it closes the circuit again.

- **Exercise 2 — Batch Error Tolerance:** Process a list of 10 documents. If any single document fails, log the error and continue processing the rest. At the end, return a report: `{processed: N, failed: M, errors: [...]}`.

- **Exercise 3 — LLM Hallucination Guard:** Add a `HallucinationGuard` to `SelfCorrectingAgent` that detects when the LLM output contains dates, names, or figures not present in the original input — and flags them as potential hallucinations before accepting the output.

- **Exercise 4 — Multi-Level Fallback with Cost Tracking:** Extend `FallbackChain` to track the cost of each attempt (e.g. precise=£0.10, general=£0.02, LLM=£0.005). Log total cost per request and cheapest successful path.

- **Exercise 5 — Async Resilience:** Rewrite `process_order` from Section 9 using `asyncio`. Run inventory check and fraud detection in parallel, but only proceed to payment if both succeed.

**Next:** Chapter 13 — Human-in-the-Loop